In [4]:
# import libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# === Load input data ===
airport_distances = pd.read_csv('../data/processed_data/nearest_airports_to_fires.csv')
airtanker_bases_distances = pd.read_csv('../data/processed_data/nearest_airtanker_bases_to_fires.csv')

# === Load GACC shapefile ===
gacc_gdf = gpd.read_file('../data/raw_data/gacc_boundaries/National_GACC_Final_20250113.shp')
gacc_gdf = gacc_gdf.rename(columns={'GACCName': 'GACC_NAME'})

# === Create GeoDataFrame: Airports ===
airport_distances = airport_distances.copy()
airport_distances['FIRE LON'] = pd.to_numeric(airport_distances['FIRE LON'], errors='coerce')
airport_distances['FIRE LAT'] = pd.to_numeric(airport_distances['FIRE LAT'], errors='coerce')
airport_distances = airport_distances.dropna(subset=['FIRE LON', 'FIRE LAT'])

airport_geometry = [Point(xy) for xy in zip(airport_distances['FIRE LON'], airport_distances['FIRE LAT'])]
airports_gdf = gpd.GeoDataFrame(airport_distances, geometry=airport_geometry, crs="EPSG:4326")

# Reproject if needed
if airports_gdf.crs != gacc_gdf.crs:
    airports_gdf = airports_gdf.to_crs(gacc_gdf.crs)

# Spatial join (keep all points)
airport_with_gacc = gpd.sjoin(
    airports_gdf,
    gacc_gdf[['geometry', 'GACC_NAME']],
    how='left',
    predicate='within'
)

# Assign Hawaii GACC if missed in spatial join
hawaii_mask = (
    (airport_with_gacc['FIRE LAT'] >= 18) & (airport_with_gacc['FIRE LAT'] <= 23) &
    (airport_with_gacc['FIRE LON'] >= -161) & (airport_with_gacc['FIRE LON'] <= -154)
)
airport_with_gacc.loc[hawaii_mask, 'GACC_NAME'] = 'Hawaii Coordination Center'

# Print summary
airport_gacc_regions = airport_with_gacc['GACC_NAME'].dropna().unique()
print(f"✈️  Airports: {len(airport_gacc_regions)} unique GACC regions found.")
print("Regions:", airport_gacc_regions)

# === Create GeoDataFrame: Airtanker Bases ===
airtanker_bases_distances = airtanker_bases_distances.copy()
airtanker_bases_distances['FIRE LON'] = pd.to_numeric(airtanker_bases_distances['FIRE LON'], errors='coerce')
airtanker_bases_distances['FIRE LAT'] = pd.to_numeric(airtanker_bases_distances['FIRE LAT'], errors='coerce')
airtanker_bases_distances = airtanker_bases_distances.dropna(subset=['FIRE LON', 'FIRE LAT'])

airtanker_geometry = [Point(xy) for xy in zip(airtanker_bases_distances['FIRE LON'], airtanker_bases_distances['FIRE LAT'])]
airtanker_bases_gdf = gpd.GeoDataFrame(airtanker_bases_distances, geometry=airtanker_geometry, crs="EPSG:4326")

# Reproject if needed
if airtanker_bases_gdf.crs != gacc_gdf.crs:
    airtanker_bases_gdf = airtanker_bases_gdf.to_crs(gacc_gdf.crs)

# Spatial join
airtanker_bases_with_gacc = gpd.sjoin(
    airtanker_bases_gdf,
    gacc_gdf[['geometry', 'GACC_NAME']],
    how='left',
    predicate='within'
)

# Assign Hawaii GACC if missed in spatial join
hawaii_mask = (
    (airtanker_bases_with_gacc['FIRE LAT'] >= 18) & (airtanker_bases_with_gacc['FIRE LAT'] <= 23) &
    (airtanker_bases_with_gacc['FIRE LON'] >= -161) & (airtanker_bases_with_gacc['FIRE LON'] <= -154)
)
airtanker_bases_with_gacc.loc[hawaii_mask, 'GACC_NAME'] = 'Hawaii Coordination Center'

# Print summary
airtanker_gacc_regions = airtanker_bases_with_gacc['GACC_NAME'].dropna().unique()
print(f"🚒 Airtanker Bases: {len(airtanker_gacc_regions)} unique GACC regions found.")
print("Regions:", airtanker_gacc_regions)

# === Save updated CSVs ===
airport_with_gacc.drop(columns='geometry').to_csv(f'../data/processed_data/nearest_airports_to_fires_with_gacc.csv', index=False)
airtanker_bases_with_gacc.drop(columns='geometry').to_csv(f'../data/processed_data/nearest_airtanker_bases_to_fires_with_gacc.csv', index=False)

print("✅ CSV files with GACC info saved.")


✈️  Airports: 11 unique GACC regions found.
Regions: ['Eastern Area Coordination Center'
 'Rocky Mountain Area Coordination Center'
 'Southern Area Coordination Center'
 'Northwest Interagency Coordination Center'
 'Northern California Geographic Area Coordination Center'
 'Great Basin Coordination Center' 'Southwest Area Coordination Center'
 'Northern Rockies Coordination Center'
 'Southern California Coordination Center'
 'Alaska Interagency Coordination Center' 'Hawaii Coordination Center']
🚒 Airtanker Bases: 11 unique GACC regions found.
Regions: ['Eastern Area Coordination Center'
 'Rocky Mountain Area Coordination Center'
 'Southern Area Coordination Center'
 'Northwest Interagency Coordination Center'
 'Northern California Geographic Area Coordination Center'
 'Great Basin Coordination Center' 'Southwest Area Coordination Center'
 'Northern Rockies Coordination Center'
 'Southern California Coordination Center'
 'Alaska Interagency Coordination Center' 'Hawaii Coordination Cent

In [5]:
# Check for any fires in Hawaii
def count_hawaii_rows(df, lat_col='FIRE LAT', lon_col='FIRE LON'):
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    hawaii_mask = (
        (df[lat_col] >= 18) & (df[lat_col] <= 23) &
        (df[lon_col] >= -161) & (df[lon_col] <= -154)
    )
    return df[hawaii_mask]

hawaii_airports = count_hawaii_rows(airport_distances)
hawaii_airtanker = count_hawaii_rows(airtanker_bases_distances)

print(f"🌺 Hawaii airport fires found: {len(hawaii_airports)}")
print(f"🌺 Hawaii airtanker base fires found: {len(hawaii_airtanker)}")


🌺 Hawaii airport fires found: 8
🌺 Hawaii airtanker base fires found: 8


In [9]:
airport_with_gacc.columns

Index(['FIRE ID', 'FIRE LAT', 'FIRE LON', 'CLOSEST_AIRPORT_NAME',
       'AIRPORT IDENT', 'RUNWAY LENGTH', 'AIRTANKER BASE', 'REGION NAME',
       'AIRPORT TYPE', 'AIRPORT LAT', 'AIRPORT LON', 'DISTANCE', 'geometry',
       'index_right', 'GACC_NAME'],
      dtype='object')

In [10]:
airtanker_bases_with_gacc.columns

Index(['FIRE ID', 'FIRE LAT', 'FIRE LON', 'CLOSEST_AIRTANKER_BASE_NAME',
       'AIRPORT IDENT', 'RUNWAY LENGTH', 'REGION NAME', 'AIRPORT TYPE',
       'AIRPORT LAT', 'AIRPORT LON', 'DISTANCE_TO_AIRTANKER_BASE', 'geometry',
       'index_right', 'GACC_NAME'],
      dtype='object')